# Steel Reinforcement Material Model

**Purpose**: Test and demonstrate the modern steel reinforcement model

**Status**: Development notebook

**Date**: January 10, 2026

## Overview

This notebook demonstrates the modern implementation of the steel reinforcement constitutive model:
- Bilinear elastic-plastic behavior with strain hardening
- Uses new `BMCSModel` base class with Pydantic validation
- Symbolic expressions with SymPy
- Automatic UI generation for interactive exploration
- Proper parameter management and caching

## Setup

Import libraries and the new steel reinforcement model.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pprint import pprint

# Import the new steel reinforcement model
from bmcs_cross_section.matmod.steel_reinforcement import SteelReinforcement, create_steel

# Try to import UI adapters
try:
    from bmcs_cross_section.core.ui.jupyter import create_interactive_plot
    JUPYTER_UI_AVAILABLE = True
    print("✅ Jupyter UI available")
except ImportError as e:
    JUPYTER_UI_AVAILABLE = False
    print(f"⚠️ Jupyter UI not available: {e}")

print("✅ Steel Reinforcement model imported successfully!")

## Test 1: Basic Model Creation

Create a steel reinforcement model with default parameters.

In [ ]:
# Create steel model with default parameters (B500A-like)
steel = SteelReinforcement()

print("Steel Reinforcement Model")
print("=" * 50)
print("\nMaterial Properties:")
pprint(steel.summary())

# Verify derived properties
print(f"\n✅ Model created successfully!")
print(f"   E_s = {steel.E_s:.0f} MPa (Young's modulus)")
print(f"   ε_sy = {steel.eps_sy:.6f} (yield strain)")
print(f"   k = {steel.ductility_ratio:.3f} (ductility ratio)")

## Test 2: Static Stress-Strain Plot

Plot the stress-strain curve without interactivity.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))
steel.plot_stress_strain(ax)
plt.tight_layout()
plt.show()

print("✅ Static plot generated!")

## Test 3: European Steel Grades

Test different European steel grades (EN 10080).

In [ ]:
# Test different steel grades
steel_grades = [
    'B500A',  # k ≥ 1.05, ε_ud ≥ 2.5%
    'B500B',  # k ≥ 1.08, ε_ud ≥ 5.0%
    'B500C',  # k ≥ 1.15, ε_ud ≥ 7.5%
]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for idx, grade in enumerate(steel_grades):
    steel_model = create_steel(grade)
    
    ax = axes[idx]
    steel_model.plot_stress_strain(ax, n_points=300)
    ax.set_title(f'{grade}: k={steel_model.ductility_ratio:.3f}, ε_ud={steel_model.eps_ud:.3f}')

plt.tight_layout()
plt.show()

print("✅ All steel grades plotted!")
print("\nKey observations:")
print("- All grades have same E_s and f_sy")
print("- Higher grade → higher ultimate strength f_st")
print("- Higher grade → larger ultimate strain ε_ud")
print("- Ductility ratio k = f_st/f_sy increases with grade")

## Test 4: Elastic-Plastic Behavior Detail

Examine the elastic and plastic branches separately.

In [ ]:
# Focus on elastic-plastic transition
steel_b500b = create_steel('B500B')

# Generate strain ranges
eps_elastic = np.linspace(-steel_b500b.eps_sy, steel_b500b.eps_sy, 100)
eps_full = np.linspace(-steel_b500b.eps_ud*1.1, steel_b500b.eps_ud*1.1, 500)

sig_elastic = steel_b500b.get_sig(eps_elastic)
sig_full = steel_b500b.get_sig(eps_full)
E_t_full = steel_b500b.get_E_t(eps_full)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Stress-strain with regions
ax1.plot(eps_full, sig_full, 'b-', linewidth=2)
ax1.axvspan(-steel_b500b.eps_sy, steel_b500b.eps_sy, alpha=0.2, color='green', 
            label='Elastic region')
ax1.axvspan(steel_b500b.eps_sy, steel_b500b.eps_ud, alpha=0.2, color='orange', 
            label='Hardening region')
ax1.axvspan(-steel_b500b.eps_ud, -steel_b500b.eps_sy, alpha=0.2, color='orange')
ax1.plot([steel_b500b.eps_sy, -steel_b500b.eps_sy],
         [steel_b500b.f_sy_scaled, -steel_b500b.f_sy_scaled],
         'ro', markersize=10, label=f'Yield: f_sy={steel_b500b.f_sy:.0f} MPa')
ax1.plot([steel_b500b.eps_ud, -steel_b500b.eps_ud],
         [steel_b500b.f_st_scaled, -steel_b500b.f_st_scaled],
         'rs', markersize=10, label=f'Ultimate: f_st={steel_b500b.f_st:.0f} MPa')
ax1.axhline(y=0, color='k', linestyle='-', linewidth=0.5)
ax1.axvline(x=0, color='k', linestyle='-', linewidth=0.5)
ax1.set_xlabel('Strain ε [-]')
ax1.set_ylabel('Stress σ [MPa]')
ax1.set_title('Steel B500B: Elastic-Plastic Behavior')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Tangent modulus
ax2.plot(eps_full, E_t_full, 'g-', linewidth=2)
ax2.axhline(y=0, color='k', linestyle='-', linewidth=0.5)
ax2.axvline(x=0, color='k', linestyle='-', linewidth=0.5)
ax2.axhline(y=steel_b500b.E_s, color='r', linestyle='--', linewidth=1,
            label=f'E_s = {steel_b500b.E_s:.0f} MPa')
ax2.set_xlabel('Strain ε [-]')
ax2.set_ylabel('Tangent Modulus E_t [MPa]')
ax2.set_title('Tangent Modulus Evolution')
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.set_ylim(-steel_b500b.E_s*0.1, steel_b500b.E_s*1.2)

plt.tight_layout()
plt.show()

print("✅ Elastic-plastic analysis complete!")
print(f"\nKey Parameters:")
print(f"  E_s = {steel_b500b.E_s:.0f} MPa (constant in elastic region)")
print(f"  ε_sy = {steel_b500b.eps_sy:.6f} (yield strain)")
print(f"  Hardening modulus ≈ {(steel_b500b.f_st - steel_b500b.f_sy)/(steel_b500b.eps_ud - steel_b500b.eps_sy):.0f} MPa")

## Test 5: Effect of Safety Factor

Compare mean strength vs design strength (with safety factor).

In [ ]:
# Compare different safety factors
factors = [1.0, 1/1.15, 1/1.50]  # Mean, characteristic, design
labels = ['Mean (γ=1.0)', 'Characteristic (γ=1.15)', 'Design (γ=1.50)']
colors = ['blue', 'orange', 'red']

fig, ax = plt.subplots(figsize=(12, 7))

for factor, label, color in zip(factors, labels, colors):
    steel_model = create_steel('B500B', factor=factor)
    
    eps = np.linspace(-steel_model.eps_ud*1.1, steel_model.eps_ud*1.1, 500)
    sig = steel_model.get_sig(eps)
    
    ax.plot(eps, sig, color=color, linewidth=2, label=label)

ax.axhline(y=0, color='k', linestyle='-', linewidth=0.5)
ax.axvline(x=0, color='k', linestyle='-', linewidth=0.5)
ax.set_xlabel('Strain ε [-]')
ax.set_ylabel('Stress σ [MPa]')
ax.set_title('Steel B500B: Effect of Safety Factor')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("✅ Safety factor effect demonstrated!")
print("\nEC2 safety factors:")
print("  γ_s = 1.00: Mean strength (testing)")
print("  γ_s = 1.15: Characteristic strength")  
print("  γ_s = 1.50: Design strength (ULS)")
print("\nNote: Factor reduces stress but not strain!")

## Test 6: Parameter Validation

Test that Pydantic validation works correctly.

In [ ]:
from pydantic import ValidationError

print("Testing parameter validation...")

# Valid model
try:
    valid = SteelReinforcement(E_s=200000, f_sy=500, f_st=540)
    print("✅ Valid parameters accepted")
except ValidationError as e:
    print(f"❌ Unexpected validation error: {e}")

# Test negative E_s (should fail)
try:
    invalid = SteelReinforcement(E_s=-200000)
    print("❌ Validation failed to catch negative E_s!")
except ValidationError as e:
    print("✅ Validation caught negative E_s:")
    print(f"   {e.errors()[0]['msg']}")

# Test negative f_sy (should fail)
try:
    invalid = SteelReinforcement(f_sy=-500)
    print("❌ Validation failed to catch negative f_sy!")
except ValidationError as e:
    print("✅ Validation caught negative f_sy:")
    print(f"   {e.errors()[0]['msg']}")

# Test negative eps_ud (should fail)
try:
    invalid = SteelReinforcement(eps_ud=-0.05)
    print("❌ Validation failed to catch negative eps_ud!")
except ValidationError as e:
    print("✅ Validation caught negative eps_ud:")
    print(f"   {e.errors()[0]['msg']}")

print("\n✅ All validation tests passed!")

## Test 7: Cache Invalidation

Test that cached properties are properly invalidated when parameters change.

In [ ]:
# Create model and access cached property
steel_model = SteelReinforcement(E_s=200000, f_sy=500)
print(f"Initial ε_sy: {steel_model.eps_sy:.6f}")
print(f"Initial f_sy_scaled: {steel_model.f_sy_scaled:.1f} MPa")

# Change parameter
steel_model.update_params(f_sy=600)
print(f"\nAfter update to f_sy=600:")
print(f"New ε_sy: {steel_model.eps_sy:.6f}")
print(f"New f_sy_scaled: {steel_model.f_sy_scaled:.1f} MPa")

# Verify values changed correctly
assert steel_model.eps_sy == 600/200000, "ε_sy not updated!"
assert steel_model.f_sy_scaled == 600, "f_sy_scaled not updated!"

print("\n✅ Cache invalidation working correctly!")

## Test 8: Interactive Plot (if ipywidgets available)

Create an interactive plot with sliders for real-time parameter exploration.

In [ ]:
if JUPYTER_UI_AVAILABLE:
    # Create a fresh model for interactive exploration
    interactive_steel = SteelReinforcement(E_s=200000, f_sy=500, f_st=540, eps_ud=0.05)
    
    # Setup function (called once)
    def setup_plot(model, ax):
        """Setup axes and create artists"""
        ax.set_xlabel('Strain ε [-]')
        ax.set_ylabel('Stress σ [MPa]')
        ax.set_title('Steel Reinforcement: Interactive Exploration')
        ax.grid(True, alpha=0.3)
        ax.axhline(y=0, color='k', linestyle='-', linewidth=0.5)
        ax.axvline(x=0, color='k', linestyle='-', linewidth=0.5)
        
        # Create artists
        line, = ax.plot([], [], 'b-', linewidth=2, label='σ-ε curve')
        yield_points, = ax.plot([], [], 'ro', markersize=10, label='Yield')
        ult_points, = ax.plot([], [], 'rs', markersize=8, label='Ultimate')
        ax.legend()
        
        return {'line': line, 'yield': yield_points, 'ultimate': ult_points}
    
    # Update function (called on parameter change)
    def update_plot(model, ax, artists):
        """Update only the data"""
        # Generate data
        eps_min, eps_max = model.get_plot_range()
        eps = np.linspace(eps_min, eps_max, 500)
        sig = model.get_sig(eps)
        
        # Update line
        artists['line'].set_data(eps, sig)
        
        # Update key points
        artists['yield'].set_data([model.eps_sy, -model.eps_sy],
                                   [model.f_sy_scaled, -model.f_sy_scaled])
        artists['ultimate'].set_data([model.eps_ud, -model.eps_ud],
                                       [model.f_st_scaled, -model.f_st_scaled])
        
        # Update limits
        ax.set_xlim(eps_min, eps_max)
        ax.relim()
        ax.autoscale_view(scalex=False)
    
    # Create interactive plot
    create_interactive_plot(
        interactive_steel,
        setup_plot,
        update_plot,
        figsize=(12, 7)
    )
    
    print("✅ Interactive plot created!")
    print("   Move the sliders to explore different parameters:")
    print("   - E_s: Young's modulus")
    print("   - f_sy: yield strength")
    print("   - f_st: ultimate strength")
    print("   - eps_ud: ultimate strain")
else:
    # Fallback: static plot
    fig, ax = plt.subplots(figsize=(12, 7))
    interactive_steel = SteelReinforcement()
    interactive_steel.plot_stress_strain(ax)
    plt.tight_layout()
    plt.show()
    print("⚠️ Static plot shown (install ipywidgets for interactive version)")

## Test 9: Comparison with Legacy Implementation

Compare results with the old implementation (if available).

In [ ]:
# Try to import legacy model for comparison
try:
    from bmcs_cross_section.matmod.reinforcement import SteelReinfMatMod
    LEGACY_AVAILABLE = True
except ImportError:
    LEGACY_AVAILABLE = False
    print("⚠️ Legacy model not available for comparison")

if LEGACY_AVAILABLE:
    # Create both models with same parameters
    new_model = SteelReinforcement(E_s=200000, f_sy=500, f_st=525, eps_ud=0.025)
    legacy_model = SteelReinfMatMod(E_s=200000, f_sy=500, f_st=525, eps_ud=0.025)
    
    # Generate test strains
    eps_test = np.linspace(-0.030, 0.030, 500)
    
    # Compute stresses
    sig_new = new_model.get_sig(eps_test)
    sig_legacy = legacy_model.get_sig(eps_test)
    
    # Plot comparison
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))
    
    # Both curves
    ax1.plot(eps_test, sig_new, 'b-', linewidth=2, label='New Implementation')
    ax1.plot(eps_test, sig_legacy, 'r--', linewidth=2, label='Legacy Implementation')
    ax1.axhline(y=0, color='k', linestyle='-', linewidth=0.5)
    ax1.axvline(x=0, color='k', linestyle='-', linewidth=0.5)
    ax1.set_xlabel('Strain ε [-]')
    ax1.set_ylabel('Stress σ [MPa]')
    ax1.set_title('Steel Model Comparison: New vs Legacy')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Difference
    diff = sig_new - sig_legacy
    ax2.plot(eps_test, diff, 'g-', linewidth=2)
    ax2.axhline(y=0, color='k', linestyle='-', linewidth=0.5)
    ax2.set_xlabel('Strain ε [-]')
    ax2.set_ylabel('Difference [MPa]')
    ax2.set_title('Stress Difference (New - Legacy)')
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Statistics
    max_diff = np.max(np.abs(diff))
    mean_diff = np.mean(np.abs(diff))
    
    print("✅ Comparison completed!")
    print(f"\nStatistics:")
    print(f"  Max absolute difference: {max_diff:.6f} MPa")
    print(f"  Mean absolute difference: {mean_diff:.6f} MPa")
    
    if max_diff < 0.01:
        print("  ✅ Excellent agreement (< 0.01 MPa)")
    elif max_diff < 0.1:
        print("  ✅ Good agreement (< 0.1 MPa)")
    else:
        print("  ⚠️ Notable differences detected")

## Summary

### What Works:
1. ✅ Steel reinforcement model with Pydantic validation
2. ✅ Bilinear elastic-plastic behavior with hardening
3. ✅ Symmetric tension-compression response
4. ✅ Symbolic expression support with SymPy
5. ✅ European steel grades (B500A/B/C)
6. ✅ Safety factor implementation
7. ✅ Cache invalidation on parameter updates
8. ✅ Interactive plotting with ipywidgets
9. ✅ Comparison with legacy implementation

### Key Features:
- **Modern architecture**: Uses `BMCSModel` base class
- **Type safety**: Pydantic validation ensures valid parameters
- **Realistic behavior**: Elastic-plastic-hardening-softening
- **Safety factors**: Support for EC2 design approach
- **UI abstraction**: Works with Jupyter, Streamlit, etc.
- **Efficient**: Cached properties, lambdified functions

### Steel Grades (EN 10080):
| Grade | f_sy [MPa] | f_st [MPa] | k [-] | ε_ud [%] |
|-------|------------|------------|-------|----------|
| B500A | 500 | 525 | 1.05 | 2.5 |
| B500B | 500 | 540 | 1.08 | 5.0 |
| B500C | 500 | 575 | 1.15 | 7.5 |

### Next Steps:
1. Create Streamlit web application
2. Integrate with EC2 concrete in cross-section analysis
3. Add more steel grades (international standards)
4. Implement temperature effects
5. Add fatigue behavior